# Train + Validate the Candidate-Filtering + BiLSTM Ensemble on Kaggle GPU

Clones the `approach/ensemble` branch and runs everything there: train the
BiLSTM fresh (same as `approach/bilstm`, which validated at 47.7% alone),
then blend it with dictionary candidate-filtering (`approach/candidate-
ngram`, 39-40% alone). Neither dominates the whole game -- candidate-
filtering gets sharp once the matching pool narrows, the BiLSTM covers
everywhere else -- so the blend weight shifts smoothly between them based
on how many dictionary words still match the current board.

**Before running:** in the notebook's Settings panel (right sidebar), set
**Accelerator = GPU T4 x2** (or any GPU) and **Internet = On** (needed to `git clone`).

In [ ]:
import torch
print('torch', torch.__version__, 'cuda available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('device:', torch.cuda.get_device_name(0))
else:
    print('WARNING: no GPU detected -- check Settings > Accelerator in the sidebar')

In [ ]:
REPO_URL = "https://github.com/Sahoo-Achyutananda/MELTWATER---HACKATHON.git"
BRANCH = "approach/ensemble"

!rm -rf repo
!git clone --branch $BRANCH --single-branch $REPO_URL repo
%cd repo/brand-buzzword-hackathon
!ls

## Use the official competition dataset

The cloned repo carries its own copy of train.txt/test.txt (downloaded from
this same competition earlier), but overwrite them here so this notebook
verifiably sources data straight from Kaggle's own `/kaggle/input/`, not an
external GitHub copy -- same content, no ambiguity for anyone reviewing it.

In [ ]:
import shutil
shutil.copy("/kaggle/input/competitions/brand-buzzword-hackathon/train.txt", "train.txt")
shutil.copy("/kaggle/input/competitions/brand-buzzword-hackathon/test.txt", "test.txt")
print("train.txt and test.txt overwritten with the official competition dataset from /kaggle/input/")

## Train the BiLSTM half of the ensemble

Same masked-language-model objective as approach/bilstm: randomly mask
letters, predict the true letter at each masked position from
bidirectional context.

In [ ]:
!python src/train_bilstm.py --epochs 20

## Validate the ensemble

Same held-out-train.txt methodology as every other branch: hold out 10%
of train.txt, play full interactive games against words the BiLSTM never
trained on. Compare directly against candidate-filtering alone (39-40%)
and BiLSTM alone (47.7%) -- this number needs to beat both to justify the
extra complexity.

In [ ]:
!python src/validate_ensemble.py

## Generate submission.csv

Plays the actual game against every word in test.txt using the ensemble
still in this session. Sandbox leaderboard checkpoint only -- per the
competition's Final Judgement policy, final hiring decisions re-run the
submitted model/notebook against a separate private word list.

250,000 words, one game at a time -- prints progress every 20,000 words
with an ETA. Expect this to run a bit slower than the BiLSTM-only
submission since every turn also does a numpy candidate-matching pass on
top of the neural forward pass.

In [ ]:
!python src/generate_submission_ensemble.py

## Save outputs

Anything under `/kaggle/working/` is downloadable from the notebook's
Output tab after the run finishes.

In [ ]:
import shutil
shutil.copy("src/bilstm_masker.pt", "/kaggle/working/bilstm_masker.pt")
shutil.copy("submission.csv", "/kaggle/working/submission.csv")
print("saved bilstm_masker.pt and submission.csv to /kaggle/working/ -- download from the Output tab")